In [1]:
import pandas as pd
import sqlite3

/root/.venv/lib/python3.12/site-packages/numpy/_core/getlimits.py:551: UserWarning: Signature b'\x00\xd0\xcc\xcc\xcc\xcc\xcc\xcc\xfb\xbf\x00\x00\x00\x00\x00\x00' for <class 'numpy.longdouble'> does not match any known type: falling back to type probe function.
This warnings indicates broken support for the dtype!
  machar = _get_machar(dtype)


### Создадим подключение к БД с помощью sqlite3

In [2]:
conn = sqlite3.connect('../data/checking-logs.sqlite')

### Создадим новую таблицу datamart, соединив таблицы pageviews и checker и используя только 1 запрос

* в таблице должны быть следующие столбцы: uid, labname, first_commit_ts, first_view_ts 
* first_commit_ts - это просто новое имя столбца timestamp из таблицы checker, оно показывает первый коммит из конкретной лаборатории и от конкретного пользователя
* first_view_ts - это первое посещение пользователем таблицы pageviews, временная метка, когда пользователь посетил ленту новостей
* status = ’ready’ по-прежнему должен быть фильтром
* numTrials = 1, по-прежнему должен быть фильтром
* labnames, имена лабораторий по-прежнему должны быть из списка: ’laba04 ’, ’laba04s’, ’laba05’, ’laba06’, ’laba06s’, ’project1’
* таблица должна содержать только пользователей (uid с user_*), а не администраторов
* first_commit_ts и first_view_ts должны быть проанализированы как datetime64[ns]

In [3]:
query = """
SELECT chk.uid,
       chk.labname,
       datetime(chk.timestamp) AS first_commit_ts,
       datetime(pv.datetime) AS first_view_ts
FROM checker AS chk
LEFT JOIN pageviews AS pv ON pv.uid=chk.uid
WHERE chk.status = 'ready' 
      AND chk.numTrials = 1
      AND chk.labname IN ('laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1')
      AND chk.uid LIKE 'user_%'
      AND (pv.datetime = (SELECT MIN(pv.datetime)
                         FROM pageviews AS pv
                         WHERE uid = chk.uid)
            OR pv.datetime IS NULL)
"""
datamart = pd.io.sql.read_sql(query, conn, parse_dates=['first_commit_ts', 'first_view_ts'])
datamart

,uid,labname,first_commit_ts,first_view_ts
0,user_4,project1,2020-04-17 05:19:02,NaT
1,user_4,laba04,2020-04-17 11:33:17,NaT
2,user_4,laba04s,2020-04-17 11:48:41,NaT
3,user_17,project1,2020-04-18 07:56:45,2020-04-18 10:56:55
4,user_30,laba04,2020-04-18 13:36:53,2020-04-17 22:46:26
...,...,...,...,...
135,user_23,laba06,2020-05-21 08:34:10,NaT
136,user_19,laba06s,2020-05-21 13:27:06,2020-04-21 20:30:38
137,user_23,laba06s,2020-05-21 14:29:15,NaT
138,user_17,laba06,2020-05-21 15:21:31,2020-04-18 10:56:55


### Создадим датафреймы test и control

* в тестовой таблице должны быть пользователи, у которых есть значения в first_view_ts
* в контрольной таблице должны быть пользователи, у которых отсутствуют значения в first_view_ts
* замените отсутствующие значения в контрольной таблице средним значением first_view_ts тестовых пользователей, мы будем использовать это значение для дальнейшего анализа
* сохраните обе таблицы в базе данных, вы будете использовать их в следующих упражнениях

In [4]:
test = datamart[datamart['first_view_ts'].notnull()]
control = datamart[datamart['first_view_ts'].isnull()]
control = control.fillna(test['first_view_ts'].mean())

In [5]:
test.head()

,uid,labname,first_commit_ts,first_view_ts
3,user_17,project1,2020-04-18 07:56:45,2020-04-18 10:56:55
4,user_30,laba04,2020-04-18 13:36:53,2020-04-17 22:46:26
7,user_30,laba04s,2020-04-18 14:51:37,2020-04-17 22:46:26
8,user_14,laba04,2020-04-18 15:14:00,2020-04-18 10:53:52
11,user_14,laba04s,2020-04-18 22:30:30,2020-04-18 10:53:52


In [6]:
control.head()

,uid,labname,first_commit_ts,first_view_ts
0,user_4,project1,2020-04-17 05:19:02,2020-04-27 00:40:05.322034176
1,user_4,laba04,2020-04-17 11:33:17,2020-04-27 00:40:05.322034176
2,user_4,laba04s,2020-04-17 11:48:41,2020-04-27 00:40:05.322034176
5,user_2,laba04,2020-04-18 13:42:35,2020-04-27 00:40:05.322034176
6,user_2,laba04s,2020-04-18 13:51:22,2020-04-27 00:40:05.322034176


In [8]:
try:
    test.to_sql('test', conn)
    control.to_sql('control', conn)
except Exception:
    print('Таблицы уже созданы')

Таблицы уже созданы


### Закроем соединение с базой данных

In [ ]:
conn.close()